## Exercise 02. Join
---
#### Task 1.

In [11]:
import pandas as pd 
import sqlite3
conn = sqlite3.connect('data/checking-logs.sqlite')

In [12]:
checker_query = """
SELECT COUNT(*) as total_count 
FROM checker 
WHERE status = 'ready'
    AND numTrials = 1
    AND labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
    AND uid LIKE 'user_%'
"""
checker_count = pd.io.sql.read_sql(checker_query, conn)
print(f"Всего записей в checker с фильтрами: {checker_count.iloc[0,0]}")

unique_pairs_query = """
SELECT COUNT(DISTINCT uid || labname) as unique_pairs
FROM checker 
WHERE status = 'ready'
    AND numTrials = 1
    AND labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
    AND uid LIKE 'user_%'
"""
unique_pairs = pd.io.sql.read_sql(unique_pairs_query, conn)
print(f"Уникальных пар (uid, labname) в checker: {unique_pairs.iloc[0,0]}")

lab_dist_query = """
SELECT labname, COUNT(*) as count
FROM checker 
WHERE status = 'ready'
    AND numTrials = 1
    AND labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
    AND uid LIKE 'user_%'
GROUP BY labname
"""
lab_dist = pd.io.sql.read_sql(lab_dist_query, conn)
print("Распределение по labname:")
print(lab_dist)

Всего записей в checker с фильтрами: 140
Уникальных пар (uid, labname) в checker: 140
Распределение по labname:
    labname  count
0    laba04     26
1   laba04s     23
2    laba05     27
3    laba06     22
4   laba06s     16
5  project1     26


In [ ]:
conn.execute("DROP TABLE IF EXISTS datamart")
conn.commit()

query = """
CREATE TABLE datamart AS
SELECT 
    c.uid,
    c.labname,
    MIN(c.timestamp) as first_commit_ts,
    MIN(p.datetime) AS first_view_ts
FROM checker c
LEFT JOIN pageviews p ON c.uid = p.uid
WHERE c.status = 'ready'
    AND c.numTrials = 1
    AND c.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
    AND c.uid LIKE 'user_%'
GROUP BY c.uid, c.labname
"""
conn.execute(query)
conn.commit()

datamart = pd.io.sql.read_sql("SELECT * FROM datamart", conn)
print(f"Строк в datamart: {len(datamart)}")

datamart['first_commit_ts'] = pd.to_datetime(datamart['first_commit_ts'])
datamart['first_view_ts'] = pd.to_datetime(datamart['first_view_ts'])

Строк в datamart: 140
Распределение по labname в datamart:
labname
laba04      26
laba04s     23
laba05      27
laba06      22
laba06s     16
project1    26
dtype: int64
NULL значений в first_commit_ts: 0


In [13]:
test = datamart[datamart['first_view_ts'].notna()].copy()
control = datamart[datamart['first_view_ts'].isna()].copy()

avg_first_view_ts = test['first_view_ts'].mean()
control['first_view_ts'] = control['first_view_ts'].fillna(avg_first_view_ts)

In [14]:
test.to_sql('test', conn, if_exists='replace', index=False)
control.to_sql('control', conn, if_exists='replace', index=False)

81

In [15]:
final_test = pd.read_sql("SELECT COUNT(*) as count FROM test", conn)
final_control = pd.read_sql("SELECT COUNT(*) as count FROM control", conn)

print(f"Итоговый test: {final_test.iloc[0, 0]} строк")
print(f"Итоговый control: {final_control.iloc[0, 0]} строк")

conn.close()

Итоговый test: 59 строк
Итоговый control: 81 строк


___